# Taxi-v3 Q-Network — Demo số liệu và liên hệ với Q-table

Notebook này được tạo từ bài `Taxi_drop-off-NN(1).ipynb`.

Mục tiêu của bản demo:
- chỉ chạy và **in số liệu**, không làm GUI;
- dùng `Taxi-v3` để cùng môi trường với demo Q-table;
- giữ đúng ý tưởng của code nguyên bản:
  - state one-hot 500 chiều;
  - mạng tuyến tính `Q(s) = sW + b`;
  - 6 Q-value ở đầu ra;
  - target `r + gamma * max(Q(s'))`;
  - loss tổng bình phương sai số;
  - Gradient Descent;
  - logic epsilon của code gốc: `p > epsilon` thì chọn action ngẫu nhiên;
- đo các chỉ số giống demo Q-table để có thể so sánh trực tiếp;
- lưu model, metrics và file kết quả.

> Code gốc dùng TensorFlow 1.x và Taxi-v2. Bản này dùng NumPy để mô phỏng đúng mạng tuyến tính và Gradient Descent của code gốc, giúp chạy dễ hơn trên Raspberry Pi hiện tại.


## 1. Import và hàm tương thích Gym/Gymnasium


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    import gymnasium as gym
    GYM_API = "gymnasium"
except ImportError:
    import gym
    GYM_API = "gym"

print("Đang dùng:", GYM_API, gym.__version__)


def reset_env(env, seed=None):
    try:
        result = env.reset(seed=seed)
    except TypeError:
        result = env.reset()

    if isinstance(result, tuple):
        return int(result[0]), result[1]

    return int(result), {}


def step_env(env, action):
    result = env.step(int(action))

    if len(result) == 5:
        s_new, reward, terminated, truncated, info = result
        return int(s_new), float(reward), bool(terminated), bool(truncated), info

    s_new, reward, done, info = result
    return int(s_new), float(reward), bool(done), False, info


## 2. Tạo Taxi-v3 — cùng môi trường với demo Q-table


In [ ]:
env = gym.make("Taxi-v3")

state, info = reset_env(env)

m = env.observation_space.n
n = env.action_space.n

print("=" * 50)
print("MÔI TRƯỜNG TAXI-v3")
print("=" * 50)
print("Initial state :", state)
print("Number states :", m)
print("Number actions:", n)
print("Q-table demo  :", (m, n))
print("Q-Network     :", f"{m} input -> {n} output")


## 3. Siêu tham số — giữ theo notebook Q-Network nguyên bản

Các giá trị dưới đây đến từ code Q-Network ban đầu:
- `gamma = 0.9`
- `max_episode = 1000`
- `max_steps = 100`
- `epsilon = 0.4`
- `alpha = 0.01`

Lưu ý: code gốc dùng `if p > epsilon: random action`, nên với `epsilon = 0.4` thì xác suất khám phá thực tế xấp xỉ **60%**.


In [ ]:
gamma = 0.9
max_episode = 1000
max_steps = 100
epsilon = 0.4
alpha = 0.01

TEST_EPISODES = 100
SEED = 42

np.random.seed(SEED)
try:
    env.action_space.seed(SEED)
except Exception:
    pass

print("gamma       =", gamma)
print("alpha       =", alpha)
print("epsilon     =", epsilon)
print("episodes    =", max_episode)
print("max_steps   =", max_steps)
print("test episodes =", TEST_EPISODES)
print("Explore thực tế theo code gốc ≈", round((1 - epsilon) * 100, 1), "%")


## 4. Q-Network tuyến tính — tương đương code TensorFlow gốc

Code gốc tạo:

`Q = sW + bias`

với:
- `s`: one-hot state, kích thước 500;
- `W`: ma trận `500 × 6`;
- `bias`: vector `6`;
- đầu ra: 6 Q-value.

Không có hidden layer, replay buffer hay target network.


In [ ]:
class QNetwork:
    def __init__(self, m, n, alpha, seed=42):
        self.m = m
        self.n = n
        self.alpha = alpha

        rng = np.random.RandomState(seed)

        # Giữ tương đương tf.random_normal(..., stddev=2)
        self.W = rng.normal(0.0, 2.0, size=(m, n))
        self.bias = rng.normal(0.0, 2.0, size=(n,))

    def get_q(self, state):
        # Với state one-hot, sW chính là hàng W[state]
        return self.W[state].copy() + self.bias.copy()

    def get_action(self, state):
        q = self.get_q(state)
        action = int(np.argmax(q))
        return action, q

    def learnQ(self, state, q_hat):
        # loss = sum((Q_hat - Q)^2)
        q = self.get_q(state)

        # dLoss/dQ = 2 * (Q - Q_hat)
        grad_q = 2.0 * (q - q_hat)

        # Vì input là one-hot:
        # chỉ hàng W[state] được cập nhật.
        self.W[state] -= self.alpha * grad_q

        # Bias dùng chung cho mọi state, giống code TensorFlow gốc.
        self.bias -= self.alpha * grad_q

        loss = np.sum((q_hat - q) ** 2)
        return float(loss)


QNN = QNetwork(m, n, alpha, seed=SEED)

print("W shape   :", QNN.W.shape)
print("Bias shape:", QNN.bias.shape)
print("Parameters:", QNN.W.size + QNN.bias.size)


## 5. Huấn luyện Q-Network

Phần này giữ đúng luồng của code gốc:

1. lấy action greedy từ Q-Network;
2. sinh `p`;
3. nếu `p > epsilon` thì ghi đè bằng action random;
4. thực hiện `env.step`;
5. tính `maxQ` ở state mới;
6. thay Q-target của action đã chọn bằng `r + gamma * maxQ`;
7. Gradient Descent một bước.

Chỉ bổ sung phần đo thời gian, reward, steps, success và loss.


In [ ]:
rewards = []
train_steps = []
train_success = []
loss_history = []

start_time = time.perf_counter()

for episode in range(max_episode):

    s, info = reset_env(env)
    rtot = 0.0
    success = False
    episode_losses = []

    for step in range(1, max_steps + 1):

        # Greedy action + Q hiện tại
        a, Q_hat = QNN.get_action(s)

        # GIỮ ĐÚNG LOGIC CODE GỐC:
        # p > epsilon thì explore ngẫu nhiên
        p = np.random.rand()

        if p > epsilon:
            a = env.action_space.sample()

        s_new, r, terminated, truncated, info = step_env(env, a)

        rtot += r

        # Q mới ở state kế tiếp
        Q_new = QNN.get_q(s_new)
        maxQ = np.max(Q_new)

        # GIỮ ĐÚNG TARGET CODE GỐC
        Q_hat[a] = r + gamma * maxQ

        loss = QNN.learnQ(s, Q_hat)
        episode_losses.append(loss)

        s = s_new

        if terminated:
            success = True

        if terminated or truncated:
            break

    rewards.append(rtot)
    train_steps.append(step)
    train_success.append(1 if success else 0)
    loss_history.append(
        float(np.mean(episode_losses)) if episode_losses else 0.0
    )

    if (episode + 1) % 100 == 0:
        print(
            f"Episode {episode+1:4d}/{max_episode} | "
            f"Avg reward last 100 = {np.mean(rewards[-100:]):8.2f} | "
            f"Avg steps last 100 = {np.mean(train_steps[-100:]):6.2f}"
        )

training_time = time.perf_counter() - start_time

print("\nTraining completed.")


## 6. Kết quả huấn luyện Q-Network


In [ ]:
avg_reward_all = float(np.mean(rewards))
avg_reward_first100 = float(np.mean(rewards[:100]))
avg_reward_last100 = float(np.mean(rewards[-100:]))
avg_steps_train = float(np.mean(train_steps))
train_success_rate = float(np.mean(train_success) * 100)
avg_loss_last100 = float(np.mean(loss_history[-100:]))

print("=" * 60)
print("Q-NETWORK - KẾT QUẢ TRAINING")
print("=" * 60)
print("Training time                 :", round(training_time, 3), "s")
print("Average reward - all          :", round(avg_reward_all, 2))
print("Average reward - first 100    :", round(avg_reward_first100, 2))
print("Average reward - last 100     :", round(avg_reward_last100, 2))
print("Average steps while training  :", round(avg_steps_train, 2))
print("Training completion rate      :", round(train_success_rate, 2), "%")
print("Average loss - last 100       :", round(avg_loss_last100, 4))


## 7. Biểu đồ reward

Biểu đồ này tương ứng trực tiếp với biểu đồ reward trong demo Q-table:
- cùng trục X là episode;
- cùng trục Y là total reward;
- dùng moving average 50 episode để nhìn xu hướng.


In [ ]:
WINDOW = 50

moving_average = np.convolve(
    rewards,
    np.ones(WINDOW) / WINDOW,
    mode="valid"
)

plt.figure(figsize=(11, 5))

plt.plot(
    range(1, max_episode + 1),
    rewards,
    alpha=0.30,
    label="Reward per episode"
)

plt.plot(
    range(WINDOW, max_episode + 1),
    moving_average,
    linewidth=2,
    label=f"Average reward ({WINDOW} episodes)"
)

plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Taxi-v3 Q-Network Training")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "qnetwork_reward.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Đã lưu: qnetwork_reward.png")


## 8. Test policy sau training — cùng cách đánh giá với Q-table

Giống phần test sau training của Q-table:
- chạy 100 episode;
- **không exploration**;
- luôn chọn `argmax(Q(state))`;
- đo:
  - số bước trung bình;
  - reward trung bình;
  - success rate;
  - min/max steps.

Đây là phần quan trọng nhất để so sánh công bằng với Q-table.


In [ ]:
test_steps = []
test_rewards = []
test_success = []

for episode in range(TEST_EPISODES):

    state, info = reset_env(env, seed=SEED + 10000 + episode)

    total_reward = 0.0
    success = False

    for step in range(1, max_steps + 1):

        # Sau training: chỉ dùng greedy policy
        q_values = QNN.get_q(state)
        action = int(np.argmax(q_values))

        state, reward, terminated, truncated, info = step_env(env, action)

        total_reward += reward

        if terminated:
            success = True

        if terminated or truncated:
            break

    test_steps.append(step)
    test_rewards.append(total_reward)
    test_success.append(1 if success else 0)


avg_test_steps = float(np.mean(test_steps))
avg_test_reward = float(np.mean(test_rewards))
success_rate_test = float(np.mean(test_success) * 100)

print("=" * 60)
print("Q-NETWORK - TEST SAU TRAINING")
print("=" * 60)
print("Test episodes        :", TEST_EPISODES)
print("Average steps        :", round(avg_test_steps, 2))
print("Average reward       :", round(avg_test_reward, 2))
print("Success rate         :", round(success_rate_test, 2), "%")
print("Minimum steps        :", int(np.min(test_steps)))
print("Maximum steps        :", int(np.max(test_steps)))


## 9. Liên hệ trực tiếp với demo Taxi Q-table

Hai demo có thể đặt cạnh nhau như sau:

| Thành phần | Q-table | Q-Network |
|---|---|---|
| Environment | Taxi-v3 | Taxi-v3 |
| State | 500 | 500 |
| Action | 6 | 6 |
| Đầu vào | state index | one-hot state 500 chiều |
| Giá trị Q | `Q[state, action]` | `sW + b` |
| Chọn action khi test | `argmax(Q[state])` | `argmax(QNN(state))` |
| Bellman target | `r + gamma*maxQ(s')` | `r + gamma*maxQ(s')` |
| Lưu kiến thức | `q_table.npy` | `qnetwork_model.npz` |
| Chỉ số đánh giá | reward, steps, success | reward, steps, success |

Điểm liên hệ quan trọng:
- cả hai đều học Q-value cho **cùng bài toán Taxi-v3**;
- cả hai đều chọn action cuối cùng bằng `argmax`;
- khác biệt chính nằm ở **cách biểu diễn Q-value**:
  - Q-table lưu trực tiếp từng giá trị;
  - Q-Network sinh Q-value từ trọng số `W` và `bias`.


In [ ]:
print("=" * 65)
print("LIÊN HỆ CẤU TRÚC Q-TABLE ↔ Q-NETWORK")
print("=" * 65)

print("Taxi-v3 states                 :", m)
print("Taxi-v3 actions                :", n)
print("Q-table entries                :", m * n)
print("Q-Network W parameters         :", QNN.W.size)
print("Q-Network bias parameters      :", QNN.bias.size)
print("Q-Network total parameters     :", QNN.W.size + QNN.bias.size)

print("\nCùng cách chọn action khi TEST:")
print("Q-table   : action = argmax(Q[state])")
print("Q-Network : action = argmax(QNetwork(state))")

print("\nCùng mục tiêu Bellman:")
print("target = reward + gamma * max(Q(next_state))")


## 10. So sánh tự động với metrics của demo Q-table

Notebook Q-table trước đó đã lưu file `qtable_all_metrics.npz`.

Nếu file đó nằm cùng thư mục với notebook này, cell dưới sẽ tự đọc và in bảng so sánh trực tiếp.

Nếu chưa có file, cell chỉ báo chưa tìm thấy — Q-Network vẫn chạy bình thường.


In [ ]:
qtable_metrics_path = "qtable_all_metrics.npz"

if os.path.exists(qtable_metrics_path):

    qt = np.load(qtable_metrics_path)

    qt_training_time = float(qt["training_time"])
    qt_avg_steps = float(qt["avg_test_steps"])
    qt_avg_reward = float(qt["avg_test_reward"])
    qt_success_rate = float(qt["success_rate_test"])

    print("=" * 78)
    print("SO SÁNH Q-TABLE VÀ Q-NETWORK TRÊN TAXI-v3")
    print("=" * 78)

    print(f"{'Metric':<28}{'Q-table':>20}{'Q-Network':>20}")
    print("-" * 78)

    print(
        f"{'Training time (s)':<28}"
        f"{qt_training_time:>20.3f}"
        f"{training_time:>20.3f}"
    )

    print(
        f"{'Average test steps':<28}"
        f"{qt_avg_steps:>20.2f}"
        f"{avg_test_steps:>20.2f}"
    )

    print(
        f"{'Average test reward':<28}"
        f"{qt_avg_reward:>20.2f}"
        f"{avg_test_reward:>20.2f}"
    )

    print(
        f"{'Test success rate (%)':<28}"
        f"{qt_success_rate:>20.2f}"
        f"{success_rate_test:>20.2f}"
    )

else:
    print(
        "Không tìm thấy qtable_all_metrics.npz.\n"
        "Hãy đặt file metrics của demo Q-table vào cùng thư mục "
        "nếu muốn so sánh tự động."
    )


## 11. Lưu model Q-Network

File này tương ứng với vai trò của `q_table.npy` trong demo Q-table.

- Q-table: lưu toàn bộ bảng Q trực tiếp.
- Q-Network: lưu `W` và `bias`.


In [ ]:
np.savez(
    "qnetwork_model.npz",
    W=QNN.W,
    bias=QNN.bias,
    state_size=m,
    action_size=n,
    gamma=gamma,
    alpha=alpha,
    epsilon=epsilon
)

print("Đã lưu: qnetwork_model.npz")
print("W shape   :", QNN.W.shape)
print("Bias shape:", QNN.bias.shape)


## 12. Lưu toàn bộ số liệu demo


In [ ]:
np.savez(
    "qnetwork_all_metrics.npz",

    # Training
    rewards=np.array(rewards),
    train_steps=np.array(train_steps),
    train_success=np.array(train_success),
    loss_history=np.array(loss_history),

    training_time=training_time,
    avg_reward_all=avg_reward_all,
    avg_reward_first100=avg_reward_first100,
    avg_reward_last100=avg_reward_last100,
    avg_steps_train=avg_steps_train,
    train_success_rate=train_success_rate,
    avg_loss_last100=avg_loss_last100,

    # Test
    test_steps=np.array(test_steps),
    test_rewards=np.array(test_rewards),
    test_success=np.array(test_success),

    avg_test_steps=avg_test_steps,
    avg_test_reward=avg_test_reward,
    success_rate_test=success_rate_test,

    # Hyperparameters
    gamma=gamma,
    alpha=alpha,
    epsilon=epsilon,
    max_episode=max_episode,
    max_steps=max_steps,
    test_episodes=TEST_EPISODES
)

print("Đã lưu: qnetwork_all_metrics.npz")


## 13. File tổng hợp kết quả để chụp màn hình khi demo


In [ ]:
summary = f"""TAXI-v3 Q-NETWORK DEMO RESULTS
============================================================

ENVIRONMENT
States: {m}
Actions: {n}

MODEL
Input: one-hot {m}
Output: {n} Q-values
W shape: {QNN.W.shape}
Bias shape: {QNN.bias.shape}
Total parameters: {QNN.W.size + QNN.bias.size}

HYPERPARAMETERS
Gamma: {gamma}
Alpha: {alpha}
Epsilon: {epsilon}
Training episodes: {max_episode}
Max steps: {max_steps}

TRAINING RESULTS
Training time: {training_time:.3f} s
Average reward all: {avg_reward_all:.2f}
Average reward first 100: {avg_reward_first100:.2f}
Average reward last 100: {avg_reward_last100:.2f}
Average training steps: {avg_steps_train:.2f}
Training completion rate: {train_success_rate:.2f} %
Average loss last 100: {avg_loss_last100:.4f}

POST-TRAINING TEST
Test episodes: {TEST_EPISODES}
Average steps: {avg_test_steps:.2f}
Average reward: {avg_test_reward:.2f}
Success rate: {success_rate_test:.2f} %
Minimum steps: {int(np.min(test_steps))}
Maximum steps: {int(np.max(test_steps))}

RELATION TO Q-TABLE DEMO
Both use Taxi-v3: 500 states, 6 actions.
Both choose greedy action by argmax during testing.
Both use Bellman target reward + gamma * max Q(next_state).
Q-table stores Q values directly.
Q-Network calculates Q values from W and bias.
============================================================
"""

print(summary)

with open(
    "qnetwork_results_summary.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(summary)

print("Đã lưu: qnetwork_results_summary.txt")
